In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
약효분류군별 성장 분석 파이프라인 v2  (YoY 기반 EI + 장단기 성장 + 리스크)
────────────────────────────────────────────────────────────────────────
입력 : meft_2020-2025.csv (월별 · 약효분류별 수량/사용금액)

핵심 지표
  · YoY_cls, YoY_mkt          : 전년 대비 성장률 (모든 EI의 기반)
  · EI_t  = (1+YoY_cls)/(1+YoY_mkt) × 100      (100 = 시장과 같은 속도)
  · EI_3Y = 최근 3개년 가중평균 EI (0.2 / 0.3 / 0.5)
  · 성장기여도(%p) = 분류군 증가액 ÷ 시작연도 전체시장 × 100  (합계 = 시장 성장률)
  · 세그먼트 = EI_3Y(≥100) × 점유율(MS, 중앙값 기준) 4분면

출력 (--out 폴더)
  1_matrix_A_promising.png     Matrix A : MS × 3Y 가중 EI  (유망성장군 발굴)
  2_matrix_B_drivers.png       Matrix B : 성장기여도 × 장기 CAGR (성장견인군)
  3_yoy_heatmap_pool.png       POOL(핵심주도+유망성장 전체) 연도별 YoY 히트맵 — 성장 패턴 일관성
  4_ei_trend_pool.png          POOL 전체 EI 추세(소형 다중) — 시장 대비 성장 지속성
  5_log_ttm_pool.png           POOL 전체 Log-TTM 매출 궤적(소형 다중) — 최근 성장세 유지/가속 여부
  A1~A3_*.png                  부록: 장·단기 CAGR 매트릭스 / 규모 상위 히트맵 / 성장 vs MDD
  growth_report.xlsx / .csv    통합 엑셀 리포트(지표·세그먼트·EI·YoY·차트 포함)

실행 : python atc_growth_analysis.py --csv meft_2020-2025.csv --out output
필요 : pandas, numpy, matplotlib, openpyxl (차트 삽입 시 pillow)
"""
import argparse
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")


In [1]:
# screen.py
# 구조적 성장 약효분류군 스크리닝
# 입력: meft_2014-2025.csv
# 출력: report_1차.md, screen_result.csv

from __future__ import annotations
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

# ----------------------------------------------------------------------
# 0. 설정
# ----------------------------------------------------------------------
CSV_PATH = Path("meft_2014-2025.csv")
OUT_MD   = Path("report_1차.md")
OUT_CSV  = Path("screen_result.csv")

# 상위 집계 행 / 무효 코드
EXCLUDE_CODES = {
    110, 140, 230, 250, 310, 398,       # 대분류 집계
    999,                                 # 해당사항없음
}

# 구조적 break 발생 코드 (해당 시점 이후만 사용)
def detect_breaks(df, threshold=0.5):
    """자동 break 탐지"""
    breaks = {}
    for code, g in df.groupby("code"):
        g = g.sort_values("date").copy()
        g["mom"] = g["amount"].pct_change()
        cand = g[g["mom"].abs() > threshold]
        if len(cand) > 0:
            breaks[code] = cand["date"].dt.strftime("%Y-%m").tolist()
    return breaks

# 자동 + 수동 병합
AUTO_BREAKS = detect_breaks(df)
MANUAL_BREAKS =  {
    111: "2017-01",   # 전신마취제 수량 급감
    116: "2017-01",   # 진훈제 재분류
    235: "2017-02",   # 최토제·진토제
    332: "2016-04",   # 지혈제 이상치
    395: "2016-04",   # 효소제제 수량
    634: "2017-01",   # 혈액제제류
}
BREAK_DATES = {**{k: v[0] for k, v in AUTO_BREAKS.items() if k not in MANUAL_BREAKS},
               **MANUAL_BREAKS}

# 중분류 매핑 (앞 2자리)
def mid_class(code: int) -> str:
    return f"{code // 10}0"

# 대분류 매핑 (앞 1자리)
def big_class(code: int) -> str:
    return f"{code // 100}00"

# 분석 창 (년)
WINDOWS = [3, 5, 7, 10, 12]

# 컷오프
MIN_AMOUNT_12M = 5_000.0        # 백만원
VOLUME_SHARE_MIN = 0.50
R2_MIN = 0.70
CV_MAX = 0.30
MDD_MIN = -0.15
NEG_YOY_MAX = 0.20
SPIKE_MAX = 0.50

# 전체 시장 CAGR (스크리닝 결과 기준, 필요시 자동 산출)
MARKET_CAGR_10Y = 0.055


# ----------------------------------------------------------------------
# 1. 데이터 로드 + 정제
# ----------------------------------------------------------------------
def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]

    # 컬럼명 정규화
    rename = {
        "약효분류코드": "code",
        "약효분류명": "name",
        "진료년월": "ym",
        "연도": "year",
        "월": "month",
        "수량_천개": "qty",
        "사용금액_백만원": "amount",
    }
    df = df.rename(columns=rename)

    # 필수 컬럼 확인
    required = ["code", "name", "ym", "year", "month", "qty", "amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    # 숫자형 변환 (빈 문자열 -> NaN)
    for c in ["code", "year", "month", "qty", "amount"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # code 결측 제거
    df = df.dropna(subset=["code"])
    df["code"] = df["code"].astype(int)

    # 제외 코드
    df = df[~df["code"].isin(EXCLUDE_CODES)]

    # ym -> 날짜
    df["ym"] = df["ym"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["ym"] + "-01", errors="coerce")
    df = df.dropna(subset=["date"])

    # 결측치 처리
    df = df.dropna(subset=["qty", "amount"])

    # name 공백 정리
    df["name"] = df["name"].astype(str).str.strip()

    # 중복 제거 (동일 code, date)
    df = df.sort_values(["code", "date"]).drop_duplicates(["code", "date"])

    return df[["code", "name", "date", "year", "month", "qty", "amount"]]


# ----------------------------------------------------------------------
# 2. 12M rolling
# ----------------------------------------------------------------------
def add_rolling(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for code, g in df.groupby("code", sort=False):
        g = g.sort_values("date").copy()
        g["amount_12m"] = g["amount"].rolling(12, min_periods=12).sum()
        g["qty_12m"]    = g["qty"].rolling(12, min_periods=12).sum()
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


# ----------------------------------------------------------------------
# 3. CAGR 계산 (12M rolling 끝점 비교)
# ----------------------------------------------------------------------
def cagr(end_val: float, start_val: float, years: int) -> float:
    if start_val is None or end_val is None:
        return np.nan
    if start_val <= 0 or end_val <= 0:
        return np.nan
    return (end_val / start_val) ** (1 / years) - 1


def trailing_cagr(g: pd.DataFrame, years: int) -> dict:
    """12M rolling 기준, 마지막 시점과 years년 전 시점 비교."""
    g = g.sort_values("date").dropna(subset=["amount_12m", "qty_12m"])
    if len(g) < 12 * years + 1:
        return {"cagr_amount": np.nan, "cagr_qty": np.nan, "cagr_price": np.nan}

    end = g.iloc[-1]
    start = g.iloc[-(12 * years + 1)]

    a_end, a_start = end["amount_12m"], start["amount_12m"]
    q_end, q_start = end["qty_12m"], start["qty_12m"]

    p_end = a_end / q_end if q_end > 0 else np.nan
    p_start = a_start / q_start if q_start > 0 else np.nan

    return {
        "cagr_amount": cagr(a_end, a_start, years),
        "cagr_qty":    cagr(q_end, q_start, years),
        "cagr_price":  cagr(p_end, p_start, years),
    }


# ----------------------------------------------------------------------
# 4. 분류군별 요약 지표
# ----------------------------------------------------------------------
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for code, g in df.groupby("code"):
        g = g.sort_values("date")

        # break 이후만 사용
        if code in BREAK_DATES:
            bd = pd.to_datetime(BREAK_DATES[code] + "-01")
            g = g[g["date"] >= bd]

        name = g["name"].iloc[0]
        last = g.dropna(subset=["amount_12m"]).iloc[-1] if len(g.dropna(subset=["amount_12m"])) else None
        if last is None:
            continue

        row = {
            "code": code,
            "name": name,
            "big":  big_class(code),
            "mid":  mid_class(code),
            "amount_12m": last["amount_12m"],
            "qty_12m":    last["qty_12m"],
            "unit_price": last["amount_12m"] / last["qty_12m"] if last["qty_12m"] > 0 else np.nan,
        }

        # 창별 CAGR
        for w in WINDOWS:
            c = trailing_cagr(g, w)
            row[f"CAGR_{w}Y"]      = c["cagr_amount"]
            row[f"qty_CAGR_{w}Y"]  = c["cagr_qty"]
            row[f"price_CAGR_{w}Y"] = c["cagr_price"]

        rows.append(row)

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------
# 5. 3중 benchmark
# ----------------------------------------------------------------------
def add_benchmark(s: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()

    for w in WINDOWS:
        col = f"CAGR_{w}Y"
        if col not in s.columns:
            continue

        # b1: 중분류 peer median (leave-one-out)
        b1 = {}
        for mid, g in s.groupby("mid"):
            for code in g["code"]:
                peers = g[g["code"] != code][col].dropna()
                b1[code] = peers.median() if len(peers) > 0 else np.nan

        # b2: 대분류 CAGR (peer median)
        b2 = {}
        for big, g in s.groupby("big"):
            for code in g["code"]:
                peers = g[g["code"] != code][col].dropna()
                b2[code] = peers.median() if len(peers) > 0 else np.nan

        # b3: 전체 시장 (leave-one-out median)
        b3 = {}
        for code in s["code"]:
            peers = s[s["code"] != code][col].dropna()
            b3[code] = peers.median() if len(peers) > 0 else np.nan

        s[f"b1_{w}Y"] = s["code"].map(b1)
        s[f"b2_{w}Y"] = s["code"].map(b2)
        s[f"b3_{w}Y"] = s["code"].map(b3)
        s[f"excess_1_{w}Y"] = s[col] - s[f"b1_{w}Y"]
        s[f"excess_2_{w}Y"] = s[col] - s[f"b2_{w}Y"]
        s[f"excess_3_{w}Y"] = s[col] - s[f"b3_{w}Y"]

    return s


# ----------------------------------------------------------------------
# 6. 다중 창 통과 판정
# ----------------------------------------------------------------------
def multi_window_pass(s: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()

    # 창별 3중 초과성장 만족
    for w in WINDOWS:
        c1 = f"excess_1_{w}Y"
        c2 = f"excess_2_{w}Y"
        c3 = f"excess_3_{w}Y"
        if c1 in s.columns:
            s[f"pass_{w}Y"] = (s[c1] > 0) & (s[c2] > 0) & (s[c3] > 0)

    pass_cols = [f"pass_{w}Y" for w in WINDOWS if f"pass_{w}Y" in s.columns]
    s["n_pass_windows"] = s[pass_cols].sum(axis=1)
    s["pass_multi"] = (
        s["n_pass_windows"] >= 4
    ) & (s["amount_12m"] >= MIN_AMOUNT_12M)

    return s


# ----------------------------------------------------------------------
# 7. 수량 vs 가격 분해 (10Y)
# ----------------------------------------------------------------------
def add_volume_share(s: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()
    a = s["CAGR_10Y"]
    q = s["qty_CAGR_10Y"]
    # log 근사: ln(1+a) ≈ ln(1+q) + ln(1+p)
    # volume_share = ln(1+q) / ln(1+a)
    with np.errstate(divide="ignore", invalid="ignore"):
        num = np.log1p(q)
        den = np.log1p(a)
        s["volume_share"] = np.where(den.abs() > 1e-9, num / den, np.nan)
    s.loc[s["volume_share"] < 0, "volume_share"] = np.nan
    s.loc[s["volume_share"] > 2, "volume_share"] = np.nan
    return s


# ----------------------------------------------------------------------
# 8. 성장 기여도 (최근 3년)
# ----------------------------------------------------------------------
def add_contribution(s: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()
    contrib = {}
    for code, g in df.groupby("code"):
        g = g.sort_values("date").dropna(subset=["amount_12m"])
        if len(g) < 12 * 3 + 1:
            contrib[code] = np.nan
            continue
        end = g.iloc[-1]["amount_12m"]
        start = g.iloc[-(12 * 3 + 1)]["amount_12m"]
        contrib[code] = end - start

    total_delta = sum(v for v in contrib.values() if not np.isnan(v))
    s["delta_3Y"] = s["code"].map(contrib)
    s["contrib_3Y"] = s["delta_3Y"] / total_delta if total_delta > 0 else np.nan
    return s


# ----------------------------------------------------------------------
# 9. 궤적 안정성
# ----------------------------------------------------------------------
def stability_metrics(g: pd.DataFrame) -> dict:
    g = g.sort_values("date").dropna(subset=["amount_12m"]).copy()
    if len(g) < 24:
        return {"R2": np.nan, "CV": np.nan, "MDD": np.nan, "neg_yoy": np.nan}

    y = np.log(g["amount_12m"].values)
    x = np.arange(len(y))
    slope, intercept, r, p, se = stats.linregress(x, y)
    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    yoy = g["amount_12m"].pct_change(12).dropna()
    cv = yoy.std() / yoy.mean() if yoy.mean() != 0 else np.nan
    neg_yoy = (yoy < 0).mean()

    # MDD
    arr = g["amount_12m"].values
    peak = np.maximum.accumulate(arr)
    dd = (arr - peak) / peak
    mdd = dd.min()

    return {"R2": r2, "CV": cv, "MDD": mdd, "neg_yoy": neg_yoy}


def add_stability(s: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()
    metrics = {}
    for code, g in df.groupby("code"):
        if code in BREAK_DATES:
            bd = pd.to_datetime(BREAK_DATES[code] + "-01")
            g = g[g["date"] >= bd]
        metrics[code] = stability_metrics(g)

    for k in ["R2", "CV", "MDD", "neg_yoy"]:
        s[k] = s["code"].map(lambda c: metrics.get(c, {}).get(k, np.nan))

    s["stable"] = (
        (s["R2"] > R2_MIN) &
        (s["CV"] < CV_MAX) &
        (s["MDD"] > MDD_MIN) &
        (s["neg_yoy"] < NEG_YOY_MAX)
    )
    return s


# ----------------------------------------------------------------------
# 10. 연도 스파이크 검사
# ----------------------------------------------------------------------
def add_spike_check(s: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()
    spikes = {}
    for code, g in df.groupby("code"):
        g = g.copy()
        g["year"] = g["date"].dt.year
        yearly = g.groupby("year")["amount"].sum().sort_index()
        if len(yearly) < 3:
            spikes[code] = np.nan
            continue
        diff = yearly.diff().dropna()
        if diff.sum() <= 0:
            spikes[code] = np.nan
            continue
        spikes[code] = diff.max() / diff.sum()
    s["spike_ratio"] = s["code"].map(spikes)
    s["no_spike"] = s["spike_ratio"] < SPIKE_MAX
    return s


# ----------------------------------------------------------------------
# 11. 점수화
# ----------------------------------------------------------------------
def add_score(s: pd.DataFrame) -> pd.DataFrame:
    s = s.copy()

    def norm(series, invert=False):
        v = series.astype(float)
        vmin, vmax = v.min(), v.max()
        if pd.isna(vmin) or pd.isna(vmax) or vmax == vmin:
            return pd.Series(0.5, index=series.index)
        out = (v - vmin) / (vmax - vmin)
        return 1 - out if invert else out

    s["_s_excess"] = norm(s["excess_3_10Y"].fillna(0))
    s["_s_multi"]  = norm(s["n_pass_windows"].fillna(0))
    s["_s_vol"]    = norm(s["volume_share"].fillna(0))
    s["_s_stab"]   = norm(s["R2"].fillna(0))
    s["_s_contrib"] = norm(s["contrib_3Y"].fillna(0))

    s["score"] = (
        0.25 * s["_s_excess"] +
        0.20 * s["_s_multi"] +
        0.20 * s["_s_vol"] +
        0.15 * s["_s_stab"] +
        0.10 * s["_s_contrib"] +
        0.10 * 0.5   # tags는 수동, 기본 0.5
    ) * 100

    return s


# ----------------------------------------------------------------------
# 12. 리포트 생성
# ----------------------------------------------------------------------
def fmt(x, digits=1, pct=False):
    if pd.isna(x):
        return "-"
    if pct:
        return f"{x*100:.{digits}f}%"
    return f"{x:.{digits}f}"


def make_report(s: pd.DataFrame, path: Path):
    lines = []
    lines.append("# 구조적 성장 약효분류군 스크리닝 결과 (1차)\n")

    # 후보
    cand = s[s["pass_multi"]].sort_values("score", ascending=False)
    lines.append(f"\n## 1차 후보 (n={len(cand)})\n")
    lines.append("| 순위 | 코드 | 분류명 | 10Y CAGR | volume_share | R² | CV | MDD | contrib_3Y | score |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|---:|")
    for i, (_, r) in enumerate(cand.iterrows(), 1):
        lines.append(
            f"| {i} | {r['code']} | {r['name']} | "
            f"{fmt(r['CAGR_10Y'], pct=True)} | "
            f"{fmt(r['volume_share'], 2)} | "
            f"{fmt(r['R2'], 3)} | "
            f"{fmt(r['CV'], 3)} | "
            f"{fmt(r['MDD'], pct=True)} | "
            f"{fmt(r['contrib_3Y'], pct=True)} | "
            f"{fmt(r['score'], 0)} |"
        )

    # 안정성 통과
    stable = cand[cand["stable"]]
    lines.append(f"\n## 안정성 통과 (n={len(stable)})\n")
    for _, r in stable.iterrows():
        lines.append(f"- {r['code']} {r['name']}")

    # 물량 기반
    vol = cand[cand["volume_share"] >= VOLUME_SHARE_MIN]
    lines.append(f"\n## 물량 기반 (n={len(vol)})\n")
    for _, r in vol.iterrows():
        lines.append(f"- {r['code']} {r['name']} (share={fmt(r['volume_share'], 2)})")

    # 가격/믹스 트랙
    price = cand[cand["volume_share"] < VOLUME_SHARE_MIN]
    lines.append(f"\n## 가격/믹스 트랙 (n={len(price)})\n")
    for _, r in price.iterrows():
        lines.append(f"- {r['code']} {r['name']} (share={fmt(r['volume_share'], 2)})")

    path.write_text("\n".join(lines), encoding="utf-8")


# ----------------------------------------------------------------------
# main
# ----------------------------------------------------------------------
def main():
    print("[1] 데이터 로드")
    df = load_data(CSV_PATH)
    print(f"    rows={len(df)}, codes={df['code'].nunique()}")

    print("[2] 12M rolling")
    df = add_rolling(df)

    print("[3] 분류군별 CAGR")
    s = summarize(df)

    print("[4] 3중 benchmark")
    s = add_benchmark(s)

    print("[5] 다중 창 통과 판정")
    s = multi_window_pass(s)

    print("[6] volume_share")
    s = add_volume_share(s)

    print("[7] 최근 3년 기여도")
    s = add_contribution(s, df)

    print("[8] 궤적 안정성")
    s = add_stability(s, df)

    print("[9] 스파이크 검사")
    s = add_spike_check(s, df)

    print("[10] 점수화")
    s = add_score(s)

    # 저장
    out = s.copy()
    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[save] {OUT_CSV}")

    make_report(s, OUT_MD)
    print(f"[save] {OUT_MD}")


if __name__ == "__main__":
    main()

NameError: name 'df' is not defined